<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW11 · Session 1 · Profiling Data and Handling Missing Values

        **Course level:** developing beginner  
        **Prior learning:** validated imports and pandas cleaning foundations  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Cleaning begins with evidence. We profile structure, missingness, duplicates, categories, and ranges before choosing any treatment.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - produce a compact data-quality profile
- measure missing values by count and percentage
- distinguish row duplicates from duplicated identifiers
- choose between retaining, filling, or dropping missing data
- add missingness indicators and document cleaning decisions


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load the dirty business dataset

Preserve an untouched raw object and work on copies.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

raw = pd.read_csv(DATA_DIR / "business_sales_dirty.csv")
print(raw.head())
print("Shape:", raw.shape)


## 1. Profile structure and types

A first profile asks: How many rows and columns? Which columns exist? What types were inferred? How many values are present? What are plausible ranges and categories?


### Create a structural profile

info reveals non-null counts and types; describe includes numerical and categorical summaries.


In [ ]:
raw.info()
print(raw.describe(include="all").transpose())


### Build a reusable quality table

One row per column combines type, missing count, missing percentage, and distinct count.


In [ ]:
quality_profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_pct": raw.isna().mean().mul(100).round(1),
    "unique_count": raw.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

print(quality_profile)


## 2. Missingness is information

- MCAR: missingness is unrelated to observed or unobserved values.
- MAR: missingness can be explained by observed variables.
- MNAR: missingness depends on the missing value itself or another unobserved cause.

These are assumptions about the data-generating process, not labels that software can prove from one table.


### Compare missingness across an observed group

Different missing rates by channel would be evidence against a simple completely-random story.


In [ ]:
missing_units_by_channel = (
    raw.assign(units_missing=raw["units"].isna())
    .groupby("channel", dropna=False)["units_missing"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "missing_rate"})
)
print(missing_units_by_channel)


## 3. Duplicates and keys

An exact duplicate repeats every selected value. A duplicated business key can be more serious because two different rows may claim the same identity. Investigate both separately.


### Count exact and key duplicates

keep=False shows every member of a duplicated-key group for inspection.


In [ ]:
exact_duplicate_count = raw.duplicated().sum()
duplicated_order_rows = raw[raw.duplicated("order_id", keep=False)].sort_values("order_id")

print("Exact duplicates:", exact_duplicate_count)
print("Rows with duplicated order_id:", len(duplicated_order_rows))
print(duplicated_order_rows[["order_id", "customer_id", "units", "unit_price"]])


## 4. Validity checks

Missingness is only one quality dimension. A value can be present but impossible. Define explicit rules from business knowledge and report failures before repairing them.


### Flag range and category problems

The source includes a negative price and an unusually large unit count for teaching.


In [ ]:
invalid_price = raw["unit_price"].notna() & (raw["unit_price"] <= 0)
suspicious_units = raw["units"].notna() & ~raw["units"].between(1, 20)
known_return_values = {"Yes", "No"}
invalid_return = raw["returned"].notna() & ~raw["returned"].isin(known_return_values)

print("Invalid prices:", invalid_price.sum())
print("Suspicious units:", suspicious_units.sum())
print("Invalid return labels:", invalid_return.sum())


## 5. Choose a missing-data treatment

Options include retaining missingness, correcting from source, adding an indicator, filling with a justified value, or dropping limited observations. The choice depends on meaning, amount, mechanism, and downstream use.


### Fill a category and preserve its absence explicitly

Unknown is not the same as a known region. An explicit label preserves that distinction.


In [ ]:
cleaned = raw.copy()
cleaned["region_was_missing"] = cleaned["region"].isna()
cleaned["region"] = cleaned["region"].fillna("Unknown")

print(cleaned[["region", "region_was_missing"]].head(15))


### Fill a numeric value with a grouped median

A category-specific median can be more relevant than one global value. transform returns one aligned value per original row.


In [ ]:
cleaned["units_was_missing"] = cleaned["units"].isna()
group_median_units = cleaned.groupby("category")["units"].transform("median")
cleaned["units"] = cleaned["units"].fillna(group_median_units)

print("Missing units after fill:", cleaned["units"].isna().sum())
print(cleaned.groupby("category")["units"].median())


### Drop only when the field is essential and loss is acceptable

For this teaching workflow, price is essential for revenue. We count the cost before dropping those few rows.


In [ ]:
rows_before_price_drop = len(cleaned)
cleaned = cleaned.dropna(subset=["unit_price"]).copy()
rows_removed_for_price = rows_before_price_drop - len(cleaned)

print("Rows removed for missing price:", rows_removed_for_price)


### Remove exact duplicates and re-check

Every cleaning action is followed by a measurable verification.


In [ ]:
rows_before_deduplication = len(cleaned)
cleaned = cleaned.drop_duplicates().copy()
rows_removed_as_duplicates = rows_before_deduplication - len(cleaned)

print("Duplicates removed:", rows_removed_as_duplicates)
print("Remaining exact duplicates:", cleaned.duplicated().sum())


## Guided decision log

A decision log turns silent data manipulation into an auditable process. Record the issue, evidence, action, reason, and impact.


### Create a cleaning decision table

The log separates what was observed from what was chosen.


In [ ]:
decision_log = pd.DataFrame([
    {
        "issue": "Missing region",
        "evidence": int(raw["region"].isna().sum()),
        "action": "Fill Unknown and add indicator",
        "reason": "Preserve rows and distinguish missingness",
    },
    {
        "issue": "Missing units",
        "evidence": int(raw["units"].isna().sum()),
        "action": "Fill category median and add indicator",
        "reason": "Small amount; category context retained",
    },
    {
        "issue": "Missing unit_price",
        "evidence": int(raw["unit_price"].isna().sum()),
        "action": "Drop rows",
        "reason": "Price is essential for revenue calculation",
    },
])
print(decision_log)


## Student coding lab

Use raw for profiling and a separate copy for every cleaning attempt.


        ### Student task 1: Build a profile

        <div class="task"><strong>Your job:</strong> Create a table with dtype, present count, missing count, missing percentage, and unique count for every raw column.</div>

        **Check your work**

        - There is one row per column.
- Percentages range from 0 to 100.
- The table is sorted by missing percentage.


In [ ]:
# STUDENT TASK 1
task_profile = pd.DataFrame(index=raw.columns)
print(task_profile)


        ### Student task 2: Audit duplicates

        <div class="task"><strong>Your job:</strong> Count exact duplicates and duplicated order IDs, then display every row involved in an order-ID duplication.</div>

        **Check your work**

        - Exact and key duplication are measured separately.
- keep=False is used for inspection.
- The displayed rows are sorted by order_id.


In [ ]:
# STUDENT TASK 2
exact_count = 0
duplicated_id_count = 0
duplicated_id_rows = pd.DataFrame()
print(exact_count, duplicated_id_count)
print(duplicated_id_rows)


        ### Student task 3: Create validity flags

        <div class="task"><strong>Your job:</strong> Flag non-positive prices, units outside 1 to 20, discounts outside 0 to 0.5, and satisfaction outside 1 to 5. Summarise flag counts.</div>

        **Check your work**

        - Missing values are not automatically counted as range failures.
- between is used with suitable boundaries.
- All four counts are calculated from raw.


In [ ]:
# STUDENT TASK 3
validity_counts = {
    "invalid_price": 0,
    "invalid_units": 0,
    "invalid_discount": 0,
    "invalid_satisfaction": 0,
}
print(pd.Series(validity_counts))


        ### Student task 4: Compare filling strategies

        <div class="task"><strong>Your job:</strong> Create global-median and category-median versions of units. Print the filled values only for rows that were originally missing.</div>

        **Check your work**

        - Two different strategies are applied.
- Only originally missing rows are compared.
- The original raw object is unchanged.


In [ ]:
# STUDENT TASK 4
comparison_data = raw.copy()
originally_missing = comparison_data["units"].isna()
comparison_data["units_global"] = comparison_data["units"]
comparison_data["units_by_category"] = comparison_data["units"]
print(comparison_data.loc[originally_missing, ["category", "units_global", "units_by_category"]])


        ### Student task 5: Create an auditable cleaned copy

        <div class="task"><strong>Your job:</strong> Copy raw, add missing indicators for region and units, fill region and units, drop missing prices, remove exact duplicates, and print before/after quality measures.</div>

        **Check your work**

        - The raw table is preserved.
- Missingness indicators are Boolean.
- Before/after evidence is calculated rather than typed.



**Optional extension:** Add a decision log with one row for each treatment.


In [ ]:
# STUDENT TASK 5
task_cleaned = raw.copy()
before = {
    "rows": len(task_cleaned),
    "missing_cells": int(task_cleaned.isna().sum().sum()),
    "duplicates": int(task_cleaned.duplicated().sum()),
}
# Add the requested cleaning operations.
after = {
    "rows": len(task_cleaned),
    "missing_cells": int(task_cleaned.isna().sum().sum()),
    "duplicates": int(task_cleaned.duplicated().sum()),
}
print("Before:", before)
print("After:", after)


        ## Knowledge check

        Answer these without running new code first.

        1. Why should raw data remain unchanged?
2. How is a duplicated key different from an exact duplicate?
3. Can a dataset alone prove MCAR, MAR, or MNAR?
4. Why add a missingness indicator?
5. What should accompany every cleaning decision?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 standardises text, parses dates, repairs invalid values, scales numbers, and validates the result.
